# 08-04. Final Nested HPO on Selected External Features
## CatBoost vs MLP — Feature Set 고정 후 마지막 튜닝

### 현재까지 확정된 것

08-03에서 `S3_selected_no_economics`가 CatBoost 기준 가장 균형 잡힌 외부 Feature Set이었습니다.

```text
Exp9Cb
+
Market Value
+
Market Momentum
+
Transfer State
+
New-Team Environment

❌ Transfer Economics 제외
```

이제 **Feature 조합 탐색은 종료**합니다.

---

# 이번 Notebook의 질문

> S3 Feature Set을 고정한 상태에서 CatBoost와 MLP를
> Nested Temporal HPO로 다시 최적화하면 어디까지 개선되는가?

---

## 절대 지키는 원칙

### 고정
- 데이터: target year 2017+
- 모집단: `matched_next == True`
- Feature Set: S3 Selected External
- Target: `next_goals`
- Outer Validation: 2020-21 ~ 2023-24
- Primary metric: MAE
- MLP Loss: MSE
- Final Test: 계속 잠금

### 튜닝
CatBoost:
- depth
- learning_rate
- l2_leaf_reg
- random_strength
- bagging_temperature

MLP:
- hidden architecture
- dropout
- learning_rate
- weight_decay
- batch_size

### 일부러 하지 않음
- Feature 재선택
- Transfer Economics 복구
- Weighted MSE / SmoothL1
- Player/Team Embedding
- Target transform
- Final Test 평가

---

# Nested Walk-forward

각 Outer Fold마다:

```text
Outer Train
    ↓
최근 3개 시즌으로 Inner Walk-forward
    ↓
Optuna
    ↓
Best Hyperparameters
    ↓
Outer Train의 마지막 시즌으로
best iteration / epoch calibration
    ↓
Outer Train 전체 재학습
    ↓
Outer Validation 단 1회 평가
```

Outer Validation은 Optuna가 절대 보지 않습니다.

## 0. Trial 수

07에서는 모델별 15 trials를 사용했습니다.

이번에는 Feature Set을 이미 확정했기 때문에
조금 더 집중해서 기본값을 **20 trials/model/fold**로 둡니다.

계산량이 너무 크다면:

```python
N_TRIALS_CATBOOST = 15
N_TRIALS_MLP = 15
```

까지 낮추는 것은 괜찮습니다.

단:

- Outer Fold 수는 줄이지 않음
- Inner Fold 수는 줄이지 않음
- Outer Validation을 tuning에 사용하지 않음

이 세 가지는 유지합니다.

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

warnings.filterwarnings("ignore")

SEED = 42

N_TRIALS_CATBOOST = 20
N_TRIALS_MLP = 20

N_INNER_FOLDS = 3

CATBOOST_MAX_ITER = 2000
CATBOOST_PATIENCE = 50

MLP_MAX_EPOCHS = 120
MLP_PATIENCE = 12


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


reset_seed()

print("Seed:", SEED)
print("CatBoost trials:", N_TRIALS_CATBOOST)
print("MLP trials:", N_TRIALS_MLP)
print("Inner folds:", N_INNER_FOLDS)

## 1. 라이브러리 확인

In [ ]:
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner

    optuna.logging.set_verbosity(
        optuna.logging.WARNING
    )

    print("optuna:", optuna.__version__)

except ImportError as exc:
    raise ImportError(
        "optuna가 필요합니다. `pip install optuna` 후 "
        "커널을 재시작하세요."
    ) from exc


try:
    import catboost
    from catboost import CatBoostRegressor

    print("catboost:", catboost.__version__)

except ImportError as exc:
    raise ImportError(
        "catboost가 필요합니다."
    ) from exc


try:
    import torch
    import torch.nn as nn

    from torch.utils.data import (
        DataLoader,
        TensorDataset,
    )

    print("torch:", torch.__version__)

except ImportError as exc:
    raise ImportError(
        "PyTorch가 필요합니다."
    ) from exc


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)


def reset_torch_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

# Part A. Data

## 2. 파일 자동 탐색

필수:

```text
08_v2_preseason_player_snapshot_dev.csv
```

선택:

```text
08_03_model_revalidation_summary.csv
08_03_model_revalidation_results.csv
```

선택 파일이 있으면 마지막에 08-03 baseline과 직접 비교합니다.

In [ ]:
MANUAL_SNAPSHOT_PATH = None
MANUAL_08_03_SUMMARY_PATH = None
MANUAL_08_03_RESULTS_PATH = None

SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

SEARCH_ROOTS = list(
    dict.fromkeys(
        p.resolve()
        for p in SEARCH_ROOTS
        if p.exists()
    )
)


def find_named_file(filename):
    candidates = []

    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob(
                filename
            ):
                if path.is_file():
                    candidates.append(path)

        except (
            PermissionError,
            OSError,
        ):
            continue

    if not candidates:
        return None

    candidates.sort(
        key=lambda p: (
            len(str(p)),
            str(p),
        )
    )

    return candidates[0]


def resolve_path(
    manual,
    filename,
    required=True,
):
    if manual is not None:
        path = Path(manual)

        if not path.exists():
            raise FileNotFoundError(
                path
            )

        return path

    path = find_named_file(
        filename
    )

    if (
        required
        and path is None
    ):
        raise FileNotFoundError(
            f"{filename}을 찾지 못했습니다."
        )

    return path


SNAPSHOT_PATH = resolve_path(
    MANUAL_SNAPSHOT_PATH,
    "08_v2_preseason_player_snapshot_dev.csv",
)

SUMMARY_08_03_PATH = resolve_path(
    MANUAL_08_03_SUMMARY_PATH,
    "08_03_model_revalidation_summary.csv",
    required=False,
)

RESULTS_08_03_PATH = resolve_path(
    MANUAL_08_03_RESULTS_PATH,
    "08_03_model_revalidation_results.csv",
    required=False,
)

print("Snapshot:", SNAPSHOT_PATH)
print("08-03 summary:", SUMMARY_08_03_PATH)
print("08-03 results:", RESULTS_08_03_PATH)

In [ ]:
df = pd.read_csv(
    SNAPSHOT_PATH,
    low_memory=False,
)

print("Raw shape:", df.shape)

if SUMMARY_08_03_PATH is not None:
    previous_summary = pd.read_csv(
        SUMMARY_08_03_PATH
    )

    print("\n08-03 summary")
    display(
        previous_summary
    )

## 3. Exp9Cb Historical Feature 재생성

In [ ]:
df["season_start"] = (
    df["season"]
    .astype(str)
    .str[:4]
    .astype(int)
)

df["target_year"] = (
    df["target_season"]
    .astype(str)
    .str[:4]
    .astype(int)
)

df = (
    df
    .sort_values(
        [
            "player",
            "season_start",
        ]
    )
    .reset_index(drop=True)
)

player_group = (
    df.groupby(
        "player",
        sort=False,
    )
)

df[
    "goals_3yr_mean"
] = player_group[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            window=3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_per90_3yr_mean"
] = player_group[
    "goals_per90"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            window=3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_3yr_max"
] = player_group[
    "goals"
].transform(
        lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .max()
)

HISTORICAL_COLS = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

df[
    HISTORICAL_COLS
] = (
    df[
        HISTORICAL_COLS
    ]
    .fillna(0.0)
)

## 4. Model B 모집단 + 2017+

In [ ]:
model_df = (
    df[
        df[
            "matched_next"
        ].eq(True)
        & df[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

model_df = (
    model_df
    .sort_values(
        [
            "season_start",
            "player",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Rows:",
    len(model_df),
)

print(
    "Season range:",
    model_df[
        "season"
    ].min(),
    "~",
    model_df[
        "season"
    ].max(),
)

print(
    "Target mean:",
    model_df[
        "next_goals"
    ].mean(),
)

print(
    "Changed-team rate:",
    f"{model_df['changed_team_preseason'].mean():.2%}"
)

## 5. Final Test Lock

In [ ]:
LOCKED_TEST_INPUT_SEASON = (
    "2024-2025"
)

LOCKED_TEST_TARGET_SEASON = (
    "2025-2026"
)

assert (
    LOCKED_TEST_INPUT_SEASON
    not in set(
        model_df[
            "season"
        ].astype(str)
    )
)

print(
    "✅ Final Test remains locked:",
    LOCKED_TEST_INPUT_SEASON,
    "→",
    LOCKED_TEST_TARGET_SEASON,
)

# Part B. Final Selected Feature Set

In [ ]:
BASE_NUMERIC = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

TEAM_BASE_FEATURES = [
    "old_team_rank_pct",
    "old_team_points_per_game",
    "old_team_goal_diff_per_game",
]

EXP9CB_NUMERIC = (
    BASE_NUMERIC
    + HISTORICAL_COLS
    + TEAM_BASE_FEATURES
)

MARKET_VALUE_FEATURES = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
]

MARKET_MOMENTUM_FEATURES = [
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

TRANSFER_STATE_FEATURES = [
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "is_loan_preseason",
    "days_since_transfer",
]

NEW_TEAM_ENVIRONMENT_FEATURES = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

SELECTED_EXTERNAL = (
    MARKET_VALUE_FEATURES
    + MARKET_MOMENTUM_FEATURES
    + TRANSFER_STATE_FEATURES
    + NEW_TEAM_ENVIRONMENT_FEATURES
)

FINAL_NUMERIC = (
    EXP9CB_NUMERIC
    + SELECTED_EXTERNAL
)

FINAL_CATEGORICAL = [
    "league",
    "position_group",
]

FEATURE_COLS = (
    FINAL_NUMERIC
    + FINAL_CATEGORICAL
)

TARGET = "next_goals"

print(
    "Final numeric:",
    len(FINAL_NUMERIC),
)

print(
    "Categorical:",
    len(FINAL_CATEGORICAL),
)

print(
    "Raw feature count:",
    len(FEATURE_COLS),
)

print("\nExcluded Economics:")
for c in [
    "transfer_fee_known",
    "log_transfer_fee",
    "fee_to_transfer_market_value_ratio",
]:
    print(" -", c)

## 6. Missingness 확인

In [ ]:
missingness = (
    model_df[
        FINAL_NUMERIC
    ]
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_rate"
    )
    .to_frame()
)

display(
    missingness.head(20)
)

# Part C. Temporal Folds

In [ ]:
OUTER_VAL_SEASONS = [
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
]


def build_inner_folds(
    outer_train_df,
    n_inner_folds=N_INNER_FOLDS,
):
    seasons = sorted(
        outer_train_df[
            "season_start"
        ]
        .dropna()
        .astype(int)
        .unique()
    )

    candidate_val_seasons = [
        season
        for season in seasons
        if (
            outer_train_df[
                "season_start"
            ]
            .lt(season)
            .any()
        )
    ]

    selected_val_seasons = (
        candidate_val_seasons[
            -n_inner_folds:
        ]
    )

    folds = []

    for val_start in (
        selected_val_seasons
    ):
        train_df = (
            outer_train_df[
                outer_train_df[
                    "season_start"
                ]
                < val_start
            ]
            .copy()
        )

        val_df = (
            outer_train_df[
                outer_train_df[
                    "season_start"
                ]
                == val_start
            ]
            .copy()
        )

        if (
            train_df.empty
            or val_df.empty
        ):
            continue

        folds.append({
            "val_season_start": (
                int(val_start)
            ),
            "train_df": train_df,
            "val_df": val_df,
        })

    if len(folds) == 0:
        raise ValueError(
            "Inner Fold를 만들 수 없습니다."
        )

    return folds


for fold_id, outer_val_season in enumerate(
    OUTER_VAL_SEASONS,
    start=1,
):
    val_start = int(
        outer_val_season[:4]
    )

    outer_train_df = (
        model_df[
            model_df[
                "season_start"
            ]
            < val_start
        ]
    )

    inner_folds = (
        build_inner_folds(
            outer_train_df
        )
    )

    print(
        f"Outer Fold {fold_id} "
        f"({outer_val_season})"
    )

    for inner in inner_folds:
        print(
            "  Inner Val:",
            inner[
                "val_season_start"
            ],
            "| Train:",
            len(
                inner[
                    "train_df"
                ]
            ),
            "| Val:",
            len(
                inner[
                    "val_df"
                ]
            ),
        )

# Part D. Metrics

In [ ]:
def regression_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    return {
        "mae": (
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "rmse": (
            mean_squared_error(
                y_true,
                y_pred,
            ) ** 0.5
        ),
        "r2": (
            r2_score(
                y_true,
                y_pred,
            )
        ),
        "bias": float(
            np.mean(
                y_pred
                - y_true
            )
        ),
    }


def slice_metrics(
    y_true,
    y_pred,
    mask,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    if mask.sum() == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "bias": np.nan,
        }

    return {
        "n": int(
            mask.sum()
        ),
        "mae": (
            mean_absolute_error(
                y_true[mask],
                y_pred[mask],
            )
        ),
        "bias": float(
            np.mean(
                y_pred[mask]
                - y_true[mask]
            )
        ),
    }


def add_diagnostics(
    row,
    val_df,
    y_true,
    pred,
):
    for threshold in [
        10,
        15,
        20,
    ]:
        h = slice_metrics(
            y_true,
            pred,
            y_true >= threshold,
        )

        row[
            f"{threshold}plus_n"
        ] = h["n"]

        row[
            f"{threshold}plus_mae"
        ] = h["mae"]

        row[
            f"{threshold}plus_bias"
        ] = h["bias"]

    changed_mask = (
        val_df[
            "changed_team_preseason"
        ]
        .fillna(0)
        .astype(int)
        .eq(1)
        .to_numpy()
    )

    changed = slice_metrics(
        y_true,
        pred,
        changed_mask,
    )

    same_team = slice_metrics(
        y_true,
        pred,
        ~changed_mask,
    )

    row[
        "changed_team_n"
    ] = changed["n"]

    row[
        "changed_team_mae"
    ] = changed["mae"]

    row[
        "changed_team_bias"
    ] = changed["bias"]

    row[
        "same_team_n"
    ] = same_team["n"]

    row[
        "same_team_mae"
    ] = same_team["mae"]

    row[
        "same_team_bias"
    ] = same_team["bias"]

    return row

# Part E. CatBoost Nested HPO

## 7. CatBoost 데이터 준비

CatBoost는 numeric NaN을 native로 처리합니다.
범주형은 문자열로 유지합니다.

In [ ]:
def prepare_catboost_xy(
    data,
):
    X = data[
        FEATURE_COLS
    ].copy()

    for col in (
        FINAL_CATEGORICAL
    ):
        X[col] = (
            X[col]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
        )

    for col in (
        FINAL_NUMERIC
    ):
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce",
        )

    y = (
        data[
            TARGET
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    cat_indices = [
        X.columns.get_loc(c)
        for c in (
            FINAL_CATEGORICAL
        )
    ]

    return (
        X,
        y,
        cat_indices,
    )

## 8. CatBoost Search Space

07과 동일한 핵심 Search Space를 사용합니다.

이번에는 Feature가 달라졌기 때문에
같은 범위를 다시 탐색하는 것이 목적입니다.

In [ ]:
def suggest_catboost_params(
    trial,
):
    return {
        "depth": (
            trial.suggest_int(
                "depth",
                4,
                8,
            )
        ),

        "learning_rate": (
            trial.suggest_float(
                "learning_rate",
                1e-2,
                0.15,
                log=True,
            )
        ),

        "l2_leaf_reg": (
            trial.suggest_float(
                "l2_leaf_reg",
                1e-2,
                20.0,
                log=True,
            )
        ),

        "random_strength": (
            trial.suggest_float(
                "random_strength",
                1e-3,
                5.0,
                log=True,
            )
        ),

        "bagging_temperature": (
            trial.suggest_float(
                "bagging_temperature",
                0.0,
                2.0,
            )
        ),
    }

## 9. CatBoost Inner Objective

In [ ]:
def make_catboost_objective(
    outer_train_df,
    base_seed,
):
    inner_folds = (
        build_inner_folds(
            outer_train_df
        )
    )

    def objective(
        trial,
    ):
        params = (
            suggest_catboost_params(
                trial
            )
        )

        fold_maes = []

        for inner_idx, fold in enumerate(
            inner_folds,
            start=1,
        ):
            (
                X_train,
                y_train,
                cat_indices,
            ) = prepare_catboost_xy(
                fold[
                    "train_df"
                ]
            )

            (
                X_val,
                y_val,
                _,
            ) = prepare_catboost_xy(
                fold[
                    "val_df"
                ]
            )

            model = (
                CatBoostRegressor(
                    iterations=(
                        CATBOOST_MAX_ITER
                    ),
                    loss_function="RMSE",
                    eval_metric="MAE",
                    bootstrap_type="Bayesian",
                    random_seed=(
                        base_seed
                        + inner_idx
                    ),
                    verbose=False,
                    allow_writing_files=False,
                    **params,
                )
            )

            model.fit(
                X_train,
                y_train,
                cat_features=(
                    cat_indices
                ),
                eval_set=(
                    X_val,
                    y_val,
                ),
                early_stopping_rounds=(
                    CATBOOST_PATIENCE
                ),
                use_best_model=True,
                verbose=False,
            )

            pred = model.predict(
                X_val
            )

            fold_mae = (
                mean_absolute_error(
                    y_val,
                    pred,
                )
            )

            fold_maes.append(
                fold_mae
            )

            running_mean = float(
                np.mean(
                    fold_maes
                )
            )

            trial.report(
                running_mean,
                step=inner_idx,
            )

            if trial.should_prune():
                raise (
                    optuna.TrialPruned()
                )

        trial.set_user_attr(
            "inner_mae_std",
            float(
                np.std(
                    fold_maes
                )
            ),
        )

        trial.set_user_attr(
            "inner_maes",
            [
                float(x)
                for x in (
                    fold_maes
                )
            ],
        )

        return float(
            np.mean(
                fold_maes
            )
        )

    return objective

## 10. Best CatBoost Params → Outer Train 재학습

Best hyperparameter가 정해진 후에도
iteration 수는 고정하지 않습니다.

Outer Train의 마지막 시즌으로 best iteration을 다시 정한 뒤,
Outer Train 전체로 그 iteration만큼 재학습합니다.

In [ ]:
def find_final_catboost_iterations(
    outer_train_df,
    best_params,
    seed,
):
    calibration_val_start = (
        outer_train_df[
            "season_start"
        ].max()
    )

    cal_train = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            < calibration_val_start
        ]
        .copy()
    )

    cal_val = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            == calibration_val_start
        ]
        .copy()
    )

    (
        X_train,
        y_train,
        cat_indices,
    ) = prepare_catboost_xy(
        cal_train
    )

    (
        X_val,
        y_val,
        _,
    ) = prepare_catboost_xy(
        cal_val
    )

    model = (
        CatBoostRegressor(
            iterations=(
                CATBOOST_MAX_ITER
            ),
            loss_function="RMSE",
            eval_metric="MAE",
            bootstrap_type="Bayesian",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **best_params,
        )
    )

    model.fit(
        X_train,
        y_train,
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_val,
            y_val,
        ),
        early_stopping_rounds=(
            CATBOOST_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    return max(
        1,
        int(
            model.get_best_iteration()
        )
        + 1,
    )


def fit_catboost_best_on_outer_train(
    outer_train_df,
    outer_val_df,
    best_params,
    seed,
):
    selected_iterations = (
        find_final_catboost_iterations(
            outer_train_df,
            best_params,
            seed=seed,
        )
    )

    (
        X_train,
        y_train,
        cat_indices,
    ) = prepare_catboost_xy(
        outer_train_df
    )

    (
        X_val,
        y_val,
        _,
    ) = prepare_catboost_xy(
        outer_val_df
    )

    model = (
        CatBoostRegressor(
            iterations=(
                selected_iterations
            ),
            loss_function="RMSE",
            bootstrap_type="Bayesian",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **best_params,
        )
    )

    start = (
        time.perf_counter()
    )

    model.fit(
        X_train,
        y_train,
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    pred = model.predict(
        X_val
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    return {
        "y_true": y_val,
        "pred": pred,
        "selected_rounds": (
            selected_iterations
        ),
        "fit_seconds": elapsed,
    }

## 11. CatBoost Nested HPO 실행

In [ ]:
catboost_outer_results = []
catboost_trial_frames = []
catboost_outer_predictions = []
catboost_best_params_by_fold = {}


for fold_id, outer_val_season in enumerate(
    OUTER_VAL_SEASONS,
    start=1,
):
    outer_val_start = int(
        outer_val_season[:4]
    )

    outer_train_df = (
        model_df[
            model_df[
                "season_start"
            ]
            < outer_val_start
        ]
        .copy()
    )

    outer_val_df = (
        model_df[
            model_df[
                "season_start"
            ]
            == outer_val_start
        ]
        .copy()
    )

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"[CatBoost] Outer Fold {fold_id} | "
        f"{outer_val_season} | "
        f"Train {len(outer_train_df):,} | "
        f"Val {len(outer_val_df):,}"
    )

    print(
        "=" * 90
    )

    sampler = TPESampler(
        seed=(
            SEED
            + fold_id * 1000
        )
    )

    pruner = MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1,
    )

    study = (
        optuna.create_study(
            direction="minimize",
            sampler=sampler,
            pruner=pruner,
            study_name=(
                f"08_04_catboost_"
                f"outer_fold_{fold_id}"
            ),
        )
    )

    objective = (
        make_catboost_objective(
            outer_train_df,
            base_seed=(
                SEED
                + 10000
                + fold_id * 1000
            ),
        )
    )

    study.optimize(
        objective,
        n_trials=(
            N_TRIALS_CATBOOST
        ),
        show_progress_bar=True,
    )

    best_params = (
        study.best_params.copy()
    )

    catboost_best_params_by_fold[
        fold_id
    ] = best_params

    print(
        "Best inner MAE:",
        study.best_value,
    )

    print(
        "Best params:",
        best_params,
    )

    outer_result = (
        fit_catboost_best_on_outer_train(
            outer_train_df,
            outer_val_df,
            best_params,
            seed=(
                SEED
                + 20000
                + fold_id
            ),
        )
    )

    metrics = regression_metrics(
        outer_result[
            "y_true"
        ],
        outer_result[
            "pred"
        ],
    )

    row = {
        "model": (
            "Tuned_CatBoost_S3"
        ),
        "fold": fold_id,
        "outer_val_season": (
            outer_val_season
        ),
        "train_n": len(
            outer_train_df
        ),
        "val_n": len(
            outer_val_df
        ),
        "best_inner_mae": (
            study.best_value
        ),
        "selected_rounds": (
            outer_result[
                "selected_rounds"
            ]
        ),
        "fit_seconds": (
            outer_result[
                "fit_seconds"
            ]
        ),
        **metrics,
    }

    row = add_diagnostics(
        row,
        outer_val_df,
        outer_result[
            "y_true"
        ],
        outer_result[
            "pred"
        ],
    )

    catboost_outer_results.append(
        row
    )

    pred_df = (
        outer_val_df[
            [
                "player",
                "team",
                "league",
                "season",
                "target_season",
                "position_group",
                "goals",
                "next_goals",
                "changed_team_preseason",
                "destination_team",
                "market_value_preseason_eur",
            ]
        ]
        .copy()
    )

    pred_df[
        "model"
    ] = (
        "Tuned_CatBoost_S3"
    )

    pred_df[
        "fold"
    ] = fold_id

    pred_df[
        "prediction"
    ] = outer_result[
        "pred"
    ]

    catboost_outer_predictions.append(
        pred_df
    )

    trials_df = (
        study.trials_dataframe(
            attrs=(
                "number",
                "value",
                "params",
                "state",
                "user_attrs",
            )
        )
    )

    trials_df[
        "outer_fold"
    ] = fold_id

    trials_df[
        "outer_val_season"
    ] = outer_val_season

    catboost_trial_frames.append(
        trials_df
    )

    print(
        f"Outer MAE: "
        f"{metrics['mae']:.4f} | "
        f"R²: {metrics['r2']:.4f} | "
        f"10+: {row['10plus_mae']:.4f}"
    )


catboost_outer_results = (
    pd.DataFrame(
        catboost_outer_results
    )
)

catboost_trials = (
    pd.concat(
        catboost_trial_frames,
        ignore_index=True,
    )
)

catboost_outer_predictions = (
    pd.concat(
        catboost_outer_predictions,
        ignore_index=True,
    )
)

# Part F. MLP Nested HPO

## 12. MLP Preprocessing

External Feature에는 결측치가 있으므로:

```text
Numeric
Train median imputation
→ StandardScaler

Categorical
OneHotEncoder(handle_unknown="ignore")
```

를 사용합니다.

Imputer와 Scaler는 각 Inner/Outer Train에서만 fit됩니다.

In [ ]:
def make_mlp_preprocessor():
    numeric_pipeline = (
        Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]
        )
    )

    return ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                FINAL_NUMERIC,
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                FINAL_CATEGORICAL,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def mlp_fit_transform(
    train_df,
    eval_df,
):
    preprocessor = (
        make_mlp_preprocessor()
    )

    X_train = (
        preprocessor.fit_transform(
            train_df[
                FEATURE_COLS
            ]
        )
    )

    X_eval = (
        preprocessor.transform(
            eval_df[
                FEATURE_COLS
            ]
        )
    )

    y_train = (
        train_df[
            TARGET
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_eval = (
        eval_df[
            TARGET
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    return (
        X_train,
        X_eval,
        y_train,
        y_eval,
        preprocessor,
    )

## 13. MLP Search Space

07과 동일한 architecture 후보를 다시 사용합니다.

이번에는 입력 feature 공간이 달라졌으므로
**같은 search space에서 다른 최적점이 선택되는지** 확인하는 것이 핵심입니다.

In [ ]:
MLP_ARCHITECTURES = {
    "64_32": [
        64,
        32,
    ],

    "128_64": [
        128,
        64,
    ],

    "128_64_32": [
        128,
        64,
        32,
    ],

    "256_128_64": [
        256,
        128,
        64,
    ],
}


def suggest_mlp_params(
    trial,
):
    architecture_name = (
        trial.suggest_categorical(
            "architecture",
            list(
                MLP_ARCHITECTURES.keys()
            ),
        )
    )

    return {
        "architecture_name": (
            architecture_name
        ),

        "hidden_dims": (
            MLP_ARCHITECTURES[
                architecture_name
            ]
        ),

        "dropout": (
            trial.suggest_float(
                "dropout",
                0.0,
                0.4,
            )
        ),

        "learning_rate": (
            trial.suggest_float(
                "learning_rate",
                1e-4,
                3e-3,
                log=True,
            )
        ),

        "weight_decay": (
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-2,
                log=True,
            )
        ),

        "batch_size": (
            trial.suggest_categorical(
                "batch_size",
                [
                    32,
                    64,
                    128,
                ],
            )
        ),
    }

## 14. Flexible MLP

In [ ]:
class FlexibleMLP(
    nn.Module
):
    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout=0.0,
    ):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in (
            hidden_dims
        ):
            layers.append(
                nn.Linear(
                    prev_dim,
                    hidden_dim,
                )
            )

            layers.append(
                nn.ReLU()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(
                        dropout
                    )
                )

            prev_dim = (
                hidden_dim
            )

        layers.append(
            nn.Linear(
                prev_dim,
                1,
            )
        )

        self.net = (
            nn.Sequential(
                *layers
            )
        )

    def forward(
        self,
        x,
    ):
        return self.net(x)


def make_loader(
    X,
    y,
    batch_size,
    shuffle=False,
):
    X_t = torch.tensor(
        X,
        dtype=torch.float32,
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
    ).reshape(
        -1,
        1,
    )

    dataset = TensorDataset(
        X_t,
        y_t,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
    )

## 15. MLP Inner Fold 학습

Loss는 MSE지만,
Early Stopping과 HPO 목적함수는 **Validation MAE**입니다.

In [ ]:
def train_mlp_inner_fold(
    train_df,
    val_df,
    params,
    seed,
):
    reset_torch_seed(
        seed
    )

    (
        X_train,
        X_val,
        y_train,
        y_val,
        _,
    ) = mlp_fit_transform(
        train_df,
        val_df,
    )

    train_loader = (
        make_loader(
            X_train,
            y_train,
            batch_size=(
                params[
                    "batch_size"
                ]
            ),
            shuffle=True,
        )
    )

    model = FlexibleMLP(
        input_dim=(
            X_train.shape[1]
        ),
        hidden_dims=(
            params[
                "hidden_dims"
            ]
        ),
        dropout=(
            params[
                "dropout"
            ]
        ),
    ).to(
        DEVICE
    )

    criterion = (
        nn.MSELoss()
    )

    optimizer = (
        torch.optim.Adam(
            model.parameters(),
            lr=(
                params[
                    "learning_rate"
                ]
            ),
            weight_decay=(
                params[
                    "weight_decay"
                ]
            ),
        )
    )

    best_mae = float(
        "inf"
    )

    best_epoch = 1
    patience_count = 0

    X_val_t = torch.tensor(
        X_val,
        dtype=torch.float32,
        device=DEVICE,
    )

    for epoch in range(
        1,
        MLP_MAX_EPOCHS + 1,
    ):
        model.train()

        for X_b, y_b in (
            train_loader
        ):
            X_b = X_b.to(
                DEVICE
            )

            y_b = y_b.to(
                DEVICE
            )

            optimizer.zero_grad()

            pred = model(
                X_b
            )

            loss = criterion(
                pred,
                y_b,
            )

            loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            val_pred = (
                model(
                    X_val_t
                )
                .cpu()
                .numpy()
                .ravel()
            )

        val_mae = (
            mean_absolute_error(
                y_val,
                val_pred,
            )
        )

        if (
            val_mae
            < best_mae - 1e-8
        ):
            best_mae = (
                val_mae
            )

            best_epoch = (
                epoch
            )

            patience_count = 0

        else:
            patience_count += 1

        if (
            patience_count
            >= MLP_PATIENCE
        ):
            break

    return {
        "best_mae": (
            best_mae
        ),
        "best_epoch": (
            best_epoch
        ),
        "epochs_executed": (
            epoch
        ),
    }

## 16. MLP Optuna Objective

In [ ]:
def make_mlp_objective(
    outer_train_df,
    base_seed,
):
    inner_folds = (
        build_inner_folds(
            outer_train_df
        )
    )

    def objective(
        trial,
    ):
        params = (
            suggest_mlp_params(
                trial
            )
        )

        fold_maes = []
        best_epochs = []

        for inner_idx, fold in enumerate(
            inner_folds,
            start=1,
        ):
            result = (
                train_mlp_inner_fold(
                    fold[
                        "train_df"
                    ],
                    fold[
                        "val_df"
                    ],
                    params=params,
                    seed=(
                        base_seed
                        + inner_idx
                    ),
                )
            )

            fold_maes.append(
                result[
                    "best_mae"
                ]
            )

            best_epochs.append(
                result[
                    "best_epoch"
                ]
            )

            running_mean = float(
                np.mean(
                    fold_maes
                )
            )

            trial.report(
                running_mean,
                step=inner_idx,
            )

            if trial.should_prune():
                raise (
                    optuna.TrialPruned()
                )

        trial.set_user_attr(
            "inner_mae_std",
            float(
                np.std(
                    fold_maes
                )
            ),
        )

        trial.set_user_attr(
            "inner_maes",
            [
                float(x)
                for x in (
                    fold_maes
                )
            ],
        )

        trial.set_user_attr(
            "inner_best_epochs",
            [
                int(x)
                for x in (
                    best_epochs
                )
            ],
        )

        return float(
            np.mean(
                fold_maes
            )
        )

    return objective

## 17. Best MLP Params → Outer Train 재학습

In [ ]:
def find_final_mlp_epoch(
    outer_train_df,
    best_params,
    seed,
):
    calibration_val_start = (
        outer_train_df[
            "season_start"
        ].max()
    )

    cal_train = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            < calibration_val_start
        ]
        .copy()
    )

    cal_val = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            == calibration_val_start
        ]
        .copy()
    )

    result = (
        train_mlp_inner_fold(
            cal_train,
            cal_val,
            params=(
                best_params
            ),
            seed=seed,
        )
    )

    return int(
        result[
            "best_epoch"
        ]
    )


def fit_mlp_best_on_outer_train(
    outer_train_df,
    outer_val_df,
    best_params,
    seed,
):
    selected_epoch = (
        find_final_mlp_epoch(
            outer_train_df,
            best_params=(
                best_params
            ),
            seed=seed,
        )
    )

    reset_torch_seed(
        seed
    )

    (
        X_train,
        X_val,
        y_train,
        y_val,
        _,
    ) = mlp_fit_transform(
        outer_train_df,
        outer_val_df,
    )

    train_loader = (
        make_loader(
            X_train,
            y_train,
            batch_size=(
                best_params[
                    "batch_size"
                ]
            ),
            shuffle=True,
        )
    )

    model = FlexibleMLP(
        input_dim=(
            X_train.shape[1]
        ),
        hidden_dims=(
            best_params[
                "hidden_dims"
            ]
        ),
        dropout=(
            best_params[
                "dropout"
            ]
        ),
    ).to(
        DEVICE
    )

    criterion = (
        nn.MSELoss()
    )

    optimizer = (
        torch.optim.Adam(
            model.parameters(),
            lr=(
                best_params[
                    "learning_rate"
                ]
            ),
            weight_decay=(
                best_params[
                    "weight_decay"
                ]
            ),
        )
    )

    start = (
        time.perf_counter()
    )

    for _ in range(
        selected_epoch
    ):
        model.train()

        for X_b, y_b in (
            train_loader
        ):
            X_b = X_b.to(
                DEVICE
            )

            y_b = y_b.to(
                DEVICE
            )

            optimizer.zero_grad()

            pred = model(
                X_b
            )

            loss = criterion(
                pred,
                y_b,
            )

            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():
        pred = (
            model(
                torch.tensor(
                    X_val,
                    dtype=torch.float32,
                    device=DEVICE,
                )
            )
            .cpu()
            .numpy()
            .ravel()
        )

    elapsed = (
        time.perf_counter()
        - start
    )

    return {
        "y_true": y_val,
        "pred": pred,
        "selected_rounds": (
            selected_epoch
        ),
        "fit_seconds": (
            elapsed
        ),
        "input_dim": (
            X_train.shape[1]
        ),
    }

## 18. MLP Nested HPO 실행

In [ ]:
mlp_outer_results = []
mlp_trial_frames = []
mlp_outer_predictions = []
mlp_best_params_by_fold = {}


for fold_id, outer_val_season in enumerate(
    OUTER_VAL_SEASONS,
    start=1,
):
    outer_val_start = int(
        outer_val_season[:4]
    )

    outer_train_df = (
        model_df[
            model_df[
                "season_start"
            ]
            < outer_val_start
        ]
        .copy()
    )

    outer_val_df = (
        model_df[
            model_df[
                "season_start"
            ]
            == outer_val_start
        ]
        .copy()
    )

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"[MLP] Outer Fold {fold_id} | "
        f"{outer_val_season} | "
        f"Train {len(outer_train_df):,} | "
        f"Val {len(outer_val_df):,}"
    )

    print(
        "=" * 90
    )

    sampler = TPESampler(
        seed=(
            SEED
            + 50000
            + fold_id * 1000
        )
    )

    pruner = MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1,
    )

    study = (
        optuna.create_study(
            direction="minimize",
            sampler=sampler,
            pruner=pruner,
            study_name=(
                f"08_04_mlp_"
                f"outer_fold_{fold_id}"
            ),
        )
    )

    objective = (
        make_mlp_objective(
            outer_train_df,
            base_seed=(
                SEED
                + 60000
                + fold_id * 10000
            ),
        )
    )

    study.optimize(
        objective,
        n_trials=(
            N_TRIALS_MLP
        ),
        show_progress_bar=True,
    )

    raw_best = (
        study.best_params.copy()
    )

    best_params = {
        "architecture_name": (
            raw_best[
                "architecture"
            ]
        ),

        "hidden_dims": (
            MLP_ARCHITECTURES[
                raw_best[
                    "architecture"
                ]
            ]
        ),

        "dropout": (
            raw_best[
                "dropout"
            ]
        ),

        "learning_rate": (
            raw_best[
                "learning_rate"
            ]
        ),

        "weight_decay": (
            raw_best[
                "weight_decay"
            ]
        ),

        "batch_size": int(
            raw_best[
                "batch_size"
            ]
        ),
    }

    mlp_best_params_by_fold[
        fold_id
    ] = best_params

    print(
        "Best inner MAE:",
        study.best_value,
    )

    print(
        "Best params:",
        best_params,
    )

    outer_result = (
        fit_mlp_best_on_outer_train(
            outer_train_df,
            outer_val_df,
            best_params=(
                best_params
            ),
            seed=(
                SEED
                + 70000
                + fold_id
            ),
        )
    )

    metrics = regression_metrics(
        outer_result[
            "y_true"
        ],
        outer_result[
            "pred"
        ],
    )

    row = {
        "model": (
            "Tuned_MLP_S3"
        ),
        "fold": fold_id,
        "outer_val_season": (
            outer_val_season
        ),
        "train_n": len(
            outer_train_df
        ),
        "val_n": len(
            outer_val_df
        ),
        "best_inner_mae": (
            study.best_value
        ),
        "selected_rounds": (
            outer_result[
                "selected_rounds"
            ]
        ),
        "input_dim": (
            outer_result[
                "input_dim"
            ]
        ),
        "fit_seconds": (
            outer_result[
                "fit_seconds"
            ]
        ),
        **metrics,
    }

    row = add_diagnostics(
        row,
        outer_val_df,
        outer_result[
            "y_true"
        ],
        outer_result[
            "pred"
        ],
    )

    mlp_outer_results.append(
        row
    )

    pred_df = (
        outer_val_df[
            [
                "player",
                "team",
                "league",
                "season",
                "target_season",
                "position_group",
                "goals",
                "next_goals",
                "changed_team_preseason",
                "destination_team",
                "market_value_preseason_eur",
            ]
        ]
        .copy()
    )

    pred_df[
        "model"
    ] = (
        "Tuned_MLP_S3"
    )

    pred_df[
        "fold"
    ] = fold_id

    pred_df[
        "prediction"
    ] = outer_result[
        "pred"
    ]

    mlp_outer_predictions.append(
        pred_df
    )

    trials_df = (
        study.trials_dataframe(
            attrs=(
                "number",
                "value",
                "params",
                "state",
                "user_attrs",
            )
        )
    )

    trials_df[
        "outer_fold"
    ] = fold_id

    trials_df[
        "outer_val_season"
    ] = outer_val_season

    mlp_trial_frames.append(
        trials_df
    )

    print(
        f"Outer MAE: "
        f"{metrics['mae']:.4f} | "
        f"R²: {metrics['r2']:.4f} | "
        f"10+: {row['10plus_mae']:.4f}"
    )


mlp_outer_results = (
    pd.DataFrame(
        mlp_outer_results
    )
)

mlp_trials = (
    pd.concat(
        mlp_trial_frames,
        ignore_index=True,
    )
)

mlp_outer_predictions = (
    pd.concat(
        mlp_outer_predictions,
        ignore_index=True,
    )
)

# Part G. Final Comparison

## 19. Tuned Model Summary

In [ ]:
tuned_outer_results = pd.concat(
    [
        catboost_outer_results,
        mlp_outer_results,
    ],
    ignore_index=True,
    sort=False,
)

tuned_outer_predictions = (
    pd.concat(
        [
            catboost_outer_predictions,
            mlp_outer_predictions,
        ],
        ignore_index=True,
        sort=False,
    )
)


def summarize_models(
    results,
):
    rows = []

    for model, g in (
        results.groupby(
            "model"
        )
    ):
        rows.append({
            "model": model,

            "folds": (
                g[
                    "fold"
                ].nunique()
            ),

            "mae_mean": (
                g[
                    "mae"
                ].mean()
            ),

            "mae_std": (
                g[
                    "mae"
                ].std(
                    ddof=1
                )
            ),

            "mae_worst": (
                g[
                    "mae"
                ].max()
            ),

            "rmse_mean": (
                g[
                    "rmse"
                ].mean()
            ),

            "r2_mean": (
                g[
                    "r2"
                ].mean()
            ),

            "bias_mean": (
                g[
                    "bias"
                ].mean()
            ),

            "10plus_mae_mean": (
                g[
                    "10plus_mae"
                ].mean()
            ),

            "10plus_bias_mean": (
                g[
                    "10plus_bias"
                ].mean()
            ),

            "15plus_mae_mean": (
                g[
                    "15plus_mae"
                ].mean()
            ),

            "20plus_mae_mean": (
                g[
                    "20plus_mae"
                ].mean()
            ),

            "changed_team_mae_mean": (
                g[
                    "changed_team_mae"
                ].mean()
            ),

            "changed_team_bias_mean": (
                g[
                    "changed_team_bias"
                ].mean()
            ),

            "same_team_mae_mean": (
                g[
                    "same_team_mae"
                ].mean()
            ),

            "fit_seconds_mean": (
                g[
                    "fit_seconds"
                ].mean()
            ),
        })

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "mae_mean"
        )
        .reset_index(
            drop=True
        )
    )


tuned_summary = (
    summarize_models(
        tuned_outer_results
    )
)

tuned_summary

## 20. Fold별 비교

In [ ]:
fold_compare = (
    tuned_outer_results
    .pivot_table(
        index=(
            "outer_val_season"
        ),
        columns="model",
        values=[
            "mae",
            "rmse",
            "r2",
            "10plus_mae",
            "20plus_mae",
            "changed_team_mae",
        ],
    )
)

fold_compare

## 21. 08-03 Baseline과 직접 비교

08-03 결과가 있으면 다음 기준과 비교합니다.

주요 reference:

```text
Fixed CatBoost S3
Frozen 07 Tuned MLP S3
Frozen 07 Tuned MLP S0
```

In [ ]:
if (
    SUMMARY_08_03_PATH
    is not None
):
    previous_summary = (
        pd.read_csv(
            SUMMARY_08_03_PATH
        )
    )

    reference = (
        previous_summary[
            (
                (
                    previous_summary[
                        "model"
                    ]
                    .eq(
                        "Fixed_CatBoost"
                    )
                )
                & (
                    previous_summary[
                        "experiment"
                    ]
                    .eq(
                        "S3_selected_no_economics"
                    )
                )
            )
            |
            (
                (
                    previous_summary[
                        "model"
                    ]
                    .eq(
                        "Frozen_07_Tuned_MLP"
                    )
                )
                & (
                    previous_summary[
                        "experiment"
                    ]
                    .isin(
                        [
                            "S0_Exp9Cb",
                            "S3_selected_no_economics",
                        ]
                    )
                )
            )
        ]
        .copy()
    )

    reference[
        "label"
    ] = (
        reference[
            "model"
        ]
        + " | "
        + reference[
            "experiment"
        ]
    )

    display(
        reference[
            [
                "label",
                "mae_mean",
                "rmse_mean",
                "r2_mean",
                "bias_mean",
                "10plus_mae_mean",
                "20plus_mae_mean",
                "changed_team_mae_mean",
            ]
        ]
    )

    print(
        "\n08-04 Tuned"
    )

    display(
        tuned_summary[
            [
                "model",
                "mae_mean",
                "rmse_mean",
                "r2_mean",
                "bias_mean",
                "10plus_mae_mean",
                "20plus_mae_mean",
                "changed_team_mae_mean",
            ]
        ]
    )

else:
    print(
        "08-03 summary 없음 → "
        "Tuned model끼리만 비교합니다."
    )

# Part H. HPO Stability

## 22. CatBoost Best Params

In [ ]:
catboost_params_rows = []

for fold_id, params in (
    catboost_best_params_by_fold.items()
):
    catboost_params_rows.append({
        "fold": fold_id,
        **params,
    })

catboost_params_df = (
    pd.DataFrame(
        catboost_params_rows
    )
)

catboost_params_df

## 23. MLP Best Params

In [ ]:
mlp_params_rows = []

for fold_id, params in (
    mlp_best_params_by_fold.items()
):
    mlp_params_rows.append({
        "fold": fold_id,
        "architecture_name": (
            params[
                "architecture_name"
            ]
        ),
        "dropout": (
            params[
                "dropout"
            ]
        ),
        "learning_rate": (
            params[
                "learning_rate"
            ]
        ),
        "weight_decay": (
            params[
                "weight_decay"
            ]
        ),
        "batch_size": (
            params[
                "batch_size"
            ]
        ),
    })

mlp_params_df = (
    pd.DataFrame(
        mlp_params_rows
    )
)

mlp_params_df

## 24. Complete / Pruned Trial 수

In [ ]:
trial_state_summary = pd.concat(
    [
        (
            catboost_trials[
                "state"
            ]
            .value_counts()
            .rename(
                "n"
            )
            .reset_index()
            .rename(
                columns={
                    "index": "state"
                }
            )
            .assign(
                model=(
                    "CatBoost"
                )
            )
        ),

        (
            mlp_trials[
                "state"
            ]
            .value_counts()
            .rename(
                "n"
            )
            .reset_index()
            .rename(
                columns={
                    "index": "state"
                }
            )
            .assign(
                model="MLP"
            )
        ),
    ],
    ignore_index=True,
)

trial_state_summary

# Part I. High-scorer / Case Analysis

## 25. 대표 선수

In [ ]:
CASE_PLAYERS = [
    "Mateo Retegui",
    "Mason Greenwood",
    "Serhou Guirassy",
    "Ousmane Dembélé",
]

case_predictions = (
    tuned_outer_predictions[
        tuned_outer_predictions[
            "player"
        ].isin(
            CASE_PLAYERS
        )
    ]
    .copy()
)

case_predictions[
    "error"
] = (
    case_predictions[
        "prediction"
    ]
    - case_predictions[
        "next_goals"
    ]
)

case_predictions[
    "abs_error"
] = (
    case_predictions[
        "error"
    ].abs()
)

display(
    case_predictions[
        [
            "player",
            "season",
            "target_season",
            "team",
            "destination_team",
            "changed_team_preseason",
            "market_value_preseason_eur",
            "next_goals",
            "model",
            "prediction",
            "error",
            "abs_error",
        ]
    ]
    .sort_values(
        [
            "player",
            "season",
            "model",
        ]
    )
)

## 26. 고득점자 실제 vs 예측 평균

In [ ]:
high_scorer_rows = []

for model, g in (
    tuned_outer_predictions
    .groupby(
        "model"
    )
):
    for threshold in [
        10,
        15,
        20,
    ]:
        part = (
            g[
                g[
                    "next_goals"
                ]
                .ge(
                    threshold
                )
            ]
        )

        high_scorer_rows.append({
            "model": model,
            "threshold": (
                threshold
            ),
            "n": len(part),
            "actual_mean": (
                part[
                    "next_goals"
                ].mean()
            ),
            "pred_mean": (
                part[
                    "prediction"
                ].mean()
            ),
            "mae": (
                mean_absolute_error(
                    part[
                        "next_goals"
                    ],
                    part[
                        "prediction"
                    ],
                )
                if len(part)
                else np.nan
            ),
            "bias": (
                (
                    part[
                        "prediction"
                    ]
                    - part[
                        "next_goals"
                    ]
                ).mean()
                if len(part)
                else np.nan
            ),
        })

high_scorer_summary = (
    pd.DataFrame(
        high_scorer_rows
    )
)

high_scorer_summary

# Part J. 시각화

In [ ]:
plot_df = (
    tuned_summary
    .sort_values(
        "mae_mean"
    )
)

plt.figure(
    figsize=(8, 4)
)

plt.barh(
    plot_df[
        "model"
    ],
    plot_df[
        "mae_mean"
    ],
    xerr=plot_df[
        "mae_std"
    ],
    capsize=4,
)

plt.xlabel(
    "Walk-forward MAE "
    "(mean ± std)"
)

plt.title(
    "08-04 Final Nested HPO — Selected S3 Features"
)

plt.grid(
    axis="x",
    alpha=0.2,
)

plt.show()

# Part K. 결과 저장

In [ ]:
ARTIFACT_DIR = Path(
    "artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTER_RESULTS_PATH = (
    ARTIFACT_DIR
    / "08_04_final_hpo_outer_results.csv"
)

SUMMARY_PATH = (
    ARTIFACT_DIR
    / "08_04_final_hpo_summary.csv"
)

PREDICTIONS_PATH = (
    ARTIFACT_DIR
    / "08_04_final_hpo_predictions.csv"
)

CAT_TRIALS_PATH = (
    ARTIFACT_DIR
    / "08_04_catboost_optuna_trials.csv"
)

MLP_TRIALS_PATH = (
    ARTIFACT_DIR
    / "08_04_mlp_optuna_trials.csv"
)

CAT_PARAMS_PATH = (
    ARTIFACT_DIR
    / "08_04_catboost_best_params_by_fold.csv"
)

MLP_PARAMS_PATH = (
    ARTIFACT_DIR
    / "08_04_mlp_best_params_by_fold.csv"
)

HIGH_SCORER_PATH = (
    ARTIFACT_DIR
    / "08_04_high_scorer_summary.csv"
)


tuned_outer_results.to_csv(
    OUTER_RESULTS_PATH,
    index=False,
)

tuned_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

tuned_outer_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
)

catboost_trials.to_csv(
    CAT_TRIALS_PATH,
    index=False,
)

mlp_trials.to_csv(
    MLP_TRIALS_PATH,
    index=False,
)

catboost_params_df.to_csv(
    CAT_PARAMS_PATH,
    index=False,
)

mlp_params_df.to_csv(
    MLP_PARAMS_PATH,
    index=False,
)

high_scorer_summary.to_csv(
    HIGH_SCORER_PATH,
    index=False,
)


print("Saved:")
for path in [
    OUTER_RESULTS_PATH,
    SUMMARY_PATH,
    PREDICTIONS_PATH,
    CAT_TRIALS_PATH,
    MLP_TRIALS_PATH,
    CAT_PARAMS_PATH,
    MLP_PARAMS_PATH,
    HIGH_SCORER_PATH,
]:
    print(
        "-",
        path.resolve(),
    )

## 27. Protocol 저장

In [ ]:
HPO_PROTOCOL = {
    "stage": (
        "08-04 Final Nested HPO"
    ),

    "data_min_target_year": (
        2017
    ),

    "population": (
        "matched_next == True"
    ),

    "feature_set": (
        "S3_selected_no_economics"
    ),

    "numeric_features": (
        FINAL_NUMERIC
    ),

    "categorical_features": (
        FINAL_CATEGORICAL
    ),

    "excluded_features": [
        "transfer_fee_known",
        "log_transfer_fee",
        "fee_to_transfer_market_value_ratio",
    ],

    "outer_validation_seasons": (
        OUTER_VAL_SEASONS
    ),

    "inner_folds": (
        N_INNER_FOLDS
    ),

    "catboost_trials": (
        N_TRIALS_CATBOOST
    ),

    "mlp_trials": (
        N_TRIALS_MLP
    ),

    "primary_metric": (
        "MAE"
    ),

    "mlp_loss": (
        "MSE"
    ),

    "test_input_season": (
        LOCKED_TEST_INPUT_SEASON
    ),

    "test_target_season": (
        LOCKED_TEST_TARGET_SEASON
    ),

    "test_status": (
        "LOCKED / NOT LOADED"
    ),

    "feature_reselection": (
        "NOT ALLOWED"
    ),

    "loss_experiment": (
        "NOT INCLUDED"
    ),
}

PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "08_04_final_hpo_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        HPO_PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    PROTOCOL_PATH.resolve()
)

# 28. 결과 해석 — 직접 작성

아래는 **의도적으로 비워둡니다.**

---

## A. CatBoost

1. 08-03 Fixed CatBoost S3 MAE `≈ 2.3826`보다 개선됐는가?
2. 4개 Outer Fold 중 몇 개에서 fixed S3보다 좋아졌는가?
3. RMSE / R²도 함께 개선됐는가?
4. 10+/15+/20+ 고득점자 성능은 개선됐는가?
5. 실제 이적 선수 MAE는 개선됐는가?
6. Best parameter가 Fold마다 크게 흔들리는가?
7. 07처럼 HPO가 baseline보다 오히려 나빠지는 현상이 반복되는가?

### 내 결론

- CatBoost HPO 효과:
- 평균 MAE:
- 안정성:
- 고득점자:
- 이적 선수:
- Hyperparameter 안정성:

---

## B. MLP

1. 08-03 Frozen MLP S3 MAE `≈ 2.4507`보다 크게 회복됐는가?
2. 기존 MLP S0 MAE `≈ 2.3738`까지 회복하는가?
3. S3에서 좋아졌던 10+/20+ 고득점 성능을 유지하는가?
4. 전체 Bias는 0 근처인가?
5. Architecture가 다시 `256_128_64`에 집중되는가?
6. dropout/lr/weight_decay가 Fold마다 크게 흔들리는가?
7. best epoch가 지나치게 짧은 Fold가 반복되는가?

### 내 결론

- MLP retuning 효과:
- 전체 MAE:
- 고득점자:
- Bias:
- 학습 안정성:
- Hyperparameter 안정성:

---

## C. 최종 모델 선택

다음 세 기준을 동시에 봅니다.

### 1. 전체 예측
- MAE
- RMSE
- R²
- Worst Fold / std

### 2. Business / Football 중요 Slice
- 10+
- 15+
- 20+
- changed_team

### 3. 운영 안정성
- Fold 간 일관성
- HPO 민감도
- 계산 비용

### 최종 결론

- 전체 MAE 1위:
- RMSE/R² 1위:
- 고득점자 1위:
- 이적 선수 1위:
- 안정성 1위:
- 최종 Model B 후보:
- 보조 Model 후보:
- 추가 HPO 필요 여부:
- Feature 실험 종료 여부:

# 다음 단계

08-04 이후에는 **HPO도 종료**하는 것을 기본 원칙으로 합니다.

다음:

# 09. Two-stage Modeling

```text
Model A
P(다음 시즌 Big5 기록 존재)

Model B
E(다음 시즌 득점 | Big5 기록 존재)
```

Model A는 전체 row를 사용하고,
Model B는 현재처럼 `matched_next=True` 조건부 회귀를 사용합니다.

---

## 고득점자 문제는 별도 연구로 분리

08-04에서도 고득점 과소예측이 강하게 남는다면
더 이상 hyperparameter 문제로 보지 않습니다.

후속 후보:

- Weighted MSE
- Poisson / Count regression
- Target transform
- Playing-time × scoring-rate decomposition
- 페널티 전담
- 역할 변화
- 부상/출전 가능성
- 감독/전술 변화

즉:

```text
Feature 문제
Model hyperparameter 문제
Objective 문제
```

를 서로 섞지 않고 순서대로 검증합니다.